# Multi-Task Fine-Tuning

## Overview
Multi-task learning (MTL) trains a single model on multiple related tasks simultaneously, enabling knowledge transfer and improved generalization. This notebook covers task balancing, loss weighting strategies, and practical implementations.

**Key Topics:**
- Multi-task learning theory and benefits
- Task balancing strategies
- Loss weighting methods (uncertainty weighting, GradNorm)
- Dataset mixing ratios
- Task interference detection
- Multi-task evaluation
- Practical multi-task setups

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
import json

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Historical Context & Theory

### Evolution of Multi-Task Learning

**1990s - Early MTL:**
- Caruana (1997): "Multitask Learning" - demonstrated improved generalization
- Shared representations across related tasks
- Simple weight sharing in neural networks

**2000s - Theoretical Foundations:**
- Task relatedness analysis
- Transfer learning theory
- Multi-task kernel methods

**2010s - Deep Multi-Task Learning:**
- Multi-task CNNs for computer vision
- Hard vs. soft parameter sharing
- Task-specific layers with shared encoders

**2017-2018 - Loss Weighting Breakthroughs:**
- Kendall et al. (2018): Uncertainty weighting for multi-task learning
- Chen et al. (2018): GradNorm - gradient normalization
- Automatic task balancing methods

**2019-2020 - Transformer MTL:**
- BERT's multi-task variants (MT-DNN)
- T5: unified text-to-text framework
- Task tokens and prompting for task selection

**2021-Present - LLM Multi-Task Training:**
- Instruction tuning as multi-task learning
- FLAN, InstructGPT: diverse task mixtures
- Dynamic task sampling strategies
- Task routing and mixture-of-experts integration

### Theoretical Foundations

**Multi-Task Objective:**
$$L_{MTL} = \sum_{t=1}^T w_t \cdot L_t(\theta_{shared}, \theta_t)$$

Where:
- $T$ = number of tasks
- $w_t$ = weight for task $t$
- $L_t$ = loss for task $t$
- $\theta_{shared}$ = shared parameters
- $\theta_t$ = task-specific parameters

**Inductive Bias Transfer:**
Multi-task learning improves generalization by:
1. **Implicit data augmentation:** Task $A$ acts as regularizer for task $B$
2. **Attention focusing:** Learning shared features useful across tasks
3. **Eavesdropping:** Task $B$ overhears learning signals from task $A$
4. **Representation bias:** Shared layers learn more general representations

## 2. Mathematical Foundations

### Task Balancing Strategies

**1. Uniform Weighting:**
$$w_t = \frac{1}{T} \quad \forall t$$

Simple but often suboptimal - ignores task difficulty and scale differences.

**2. Uncertainty Weighting (Kendall et al., 2018):**
$$L_{MTL} = \sum_{t=1}^T \frac{1}{2\sigma_t^2} L_t + \log(\sigma_t)$$

Where $\sigma_t$ is learned task-dependent noise parameter. The second term prevents $\sigma_t \to \infty$.

**3. GradNorm (Chen et al., 2018):**
Balances gradient magnitudes across tasks:
$$|\nabla_{\theta_{shared}} L_t| \approx \bar{G} \times [r_t(t)]^\alpha$$

Where:
- $\bar{G}$ = average gradient norm across tasks
- $r_t(t) = \frac{\tilde{L}_t(t)}{\tilde{L}_t(0)}$ = relative loss decrease
- $\alpha$ = restoring force hyperparameter (typical: 1.5)

**4. Dynamic Task Prioritization (DTP):**
$$w_t(i) = w_t(i-1) \cdot \exp\left(\gamma \cdot \frac{L_t(i) - L_t(i-k)}{L_t(i-k)}\right)$$

Increases weight for tasks with slower learning progress.

### Dataset Mixing Ratios

**Temperature-based Sampling:**
$$P(\text{task } t) = \frac{|D_t|^\tau}{\sum_{j=1}^T |D_j|^\tau}$$

Where:
- $|D_t|$ = dataset size for task $t$
- $\tau$ = temperature (0 = uniform, 1 = proportional to size)

**Examples-proportional Mixing (T5):**
- $\tau = 1.0$: Sample proportional to dataset size
- $\tau = 0.5$: Square root sampling (balance large/small tasks)
- Artificial limits on large datasets (e.g., max 500K examples)

## 3. Implementation: Core Multi-Task Components

In [ ]:
@dataclass
class TaskConfig:
    """Configuration for a single task."""
    name: str
    loss_fn: str  # 'cross_entropy', 'mse', 'bce'
    weight: float = 1.0
    dataset_size: int = 1000
    num_classes: Optional[int] = None
    
class MultiTaskHead(nn.Module):
    """Task-specific head for multi-task learning."""
    
    def __init__(self, input_dim: int, task_configs: List[TaskConfig]):
        super().__init__()
        self.task_configs = {tc.name: tc for tc in task_configs}
        
        # Create task-specific output layers
        self.task_heads = nn.ModuleDict()
        for config in task_configs:
            if config.num_classes:
                # Classification head
                self.task_heads[config.name] = nn.Sequential(
                    nn.Linear(input_dim, input_dim // 2),
                    nn.ReLU(),
                    nn.Dropout(0.1),
                    nn.Linear(input_dim // 2, config.num_classes)
                )
            else:
                # Regression head
                self.task_heads[config.name] = nn.Sequential(
                    nn.Linear(input_dim, input_dim // 2),
                    nn.ReLU(),
                    nn.Linear(input_dim // 2, 1)
                )
    
    def forward(self, shared_repr: torch.Tensor, task_name: str) -> torch.Tensor:
        return self.task_heads[task_name](shared_repr)

class UncertaintyWeighting(nn.Module):
    """Uncertainty-based task weighting (Kendall et al., 2018)."""
    
    def __init__(self, num_tasks: int):
        super().__init__()
        # Log variance for numerical stability
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))
    
    def forward(self, losses: torch.Tensor) -> torch.Tensor:
        """
        Args:
            losses: Tensor of shape (num_tasks,) with individual task losses
        
        Returns:
            Weighted total loss
        """
        precision = torch.exp(-self.log_vars)
        weighted_loss = torch.sum(precision * losses + self.log_vars)
        return weighted_loss
    
    def get_weights(self) -> torch.Tensor:
        """Get current task weights (precision)."""
        return torch.exp(-self.log_vars)

class GradNormWeighting:
    """GradNorm task balancing (Chen et al., 2018)."""
    
    def __init__(self, num_tasks: int, alpha: float = 1.5):
        self.num_tasks = num_tasks
        self.alpha = alpha
        self.weights = torch.ones(num_tasks, requires_grad=True)
        self.initial_losses = None
        
    def compute_weights(
        self,
        losses: torch.Tensor,
        shared_params: nn.Parameter,
        learning_rate: float = 0.025
    ) -> torch.Tensor:
        """
        Compute GradNorm-balanced task weights.
        
        Args:
            losses: Current task losses
            shared_params: Shared model parameters for gradient computation
            learning_rate: Learning rate for weight updates
        """
        if self.initial_losses is None:
            self.initial_losses = losses.detach().clone()
        
        # Compute gradient norms for each task
        grad_norms = []
        for i in range(self.num_tasks):
            # Compute gradient of weighted loss w.r.t. shared params
            grad = torch.autograd.grad(
                self.weights[i] * losses[i],
                shared_params,
                retain_graph=True,
                create_graph=True
            )[0]
            grad_norms.append(grad.norm())
        
        grad_norms = torch.stack(grad_norms)
        
        # Compute inverse training rates
        loss_ratios = losses.detach() / self.initial_losses
        inverse_train_rates = loss_ratios / loss_ratios.mean()
        
        # Target gradient norms
        mean_grad = grad_norms.mean()
        target_grads = mean_grad * (inverse_train_rates ** self.alpha)
        
        # Compute GradNorm loss and update weights
        gradnorm_loss = F.l1_loss(grad_norms, target_grads)
        
        # Update weights
        weight_grad = torch.autograd.grad(gradnorm_loss, self.weights)[0]
        with torch.no_grad():
            self.weights -= learning_rate * weight_grad
            # Normalize weights
            self.weights = self.num_tasks * self.weights / self.weights.sum()
            self.weights = torch.clamp(self.weights, min=0.1, max=10.0)
        
        return self.weights.detach()

print("Multi-task components implemented")

## 4. Dataset Mixing Strategies

In [ ]:
class MultiTaskDataLoader:
    """Data loader with task mixing strategies."""
    
    def __init__(
        self,
        task_datasets: Dict[str, List],
        batch_size: int = 32,
        mixing_strategy: str = 'proportional',
        temperature: float = 1.0,
        max_examples_per_task: Optional[int] = None
    ):
        self.task_datasets = task_datasets
        self.batch_size = batch_size
        self.mixing_strategy = mixing_strategy
        self.temperature = temperature
        self.max_examples_per_task = max_examples_per_task
        
        # Compute sampling probabilities
        self.sampling_probs = self._compute_sampling_probs()
        
        # Create iterators
        self.task_iterators = {}
        self.reset_iterators()
    
    def _compute_sampling_probs(self) -> Dict[str, float]:
        """Compute task sampling probabilities based on mixing strategy."""
        task_sizes = {name: len(data) for name, data in self.task_datasets.items()}
        
        # Apply max examples limit
        if self.max_examples_per_task:
            task_sizes = {
                name: min(size, self.max_examples_per_task)
                for name, size in task_sizes.items()
            }
        
        if self.mixing_strategy == 'uniform':
            # Equal probability for all tasks
            num_tasks = len(task_sizes)
            return {name: 1.0 / num_tasks for name in task_sizes}
        
        elif self.mixing_strategy == 'proportional':
            # Temperature-scaled proportional sampling
            powered_sizes = {name: size ** self.temperature for name, size in task_sizes.items()}
            total = sum(powered_sizes.values())
            return {name: size / total for name, size in powered_sizes.items()}
        
        elif self.mixing_strategy == 'sqrt':
            # Square root sampling (T5-style)
            sqrt_sizes = {name: size ** 0.5 for name, size in task_sizes.items()}
            total = sum(sqrt_sizes.values())
            return {name: size / total for name, size in sqrt_sizes.items()}
        
        else:
            raise ValueError(f"Unknown mixing strategy: {self.mixing_strategy}")
    
    def reset_iterators(self):
        """Reset all task iterators."""
        for task_name, dataset in self.task_datasets.items():
            # Shuffle and create iterator
            indices = torch.randperm(len(dataset))
            self.task_iterators[task_name] = iter([(dataset[i], task_name) for i in indices])
    
    def sample_task(self) -> str:
        """Sample a task based on mixing probabilities."""
        tasks = list(self.sampling_probs.keys())
        probs = [self.sampling_probs[t] for t in tasks]
        return np.random.choice(tasks, p=probs)
    
    def get_batch(self) -> Tuple[List, List[str]]:
        """Get a mixed batch of examples from different tasks."""
        batch_data = []
        batch_tasks = []
        
        for _ in range(self.batch_size):
            task_name = self.sample_task()
            
            try:
                data, task = next(self.task_iterators[task_name])
                batch_data.append(data)
                batch_tasks.append(task)
            except StopIteration:
                # Reset iterator if exhausted
                dataset = self.task_datasets[task_name]
                indices = torch.randperm(len(dataset))
                self.task_iterators[task_name] = iter(
                    [(dataset[i], task_name) for i in indices]
                )
                data, task = next(self.task_iterators[task_name])
                batch_data.append(data)
                batch_tasks.append(task)
        
        return batch_data, batch_tasks

# Visualize mixing strategies
def visualize_mixing_strategies():
    """Compare different dataset mixing strategies."""
    # Simulated task sizes (imbalanced)
    task_sizes = {
        'Task A': 100000,
        'Task B': 10000,
        'Task C': 1000,
        'Task D': 500,
        'Task E': 100
    }
    
    strategies = ['uniform', 'proportional', 'sqrt']
    temperatures = [0.5, 1.0, 1.5]  # For proportional strategy
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, strategy in enumerate(strategies):
        ax = axes[idx]
        
        if strategy == 'proportional':
            # Show multiple temperatures
            for temp in temperatures:
                loader = MultiTaskDataLoader(
                    {name: list(range(size)) for name, size in task_sizes.items()},
                    mixing_strategy=strategy,
                    temperature=temp
                )
                probs = [loader.sampling_probs[t] * 100 for t in task_sizes.keys()]
                ax.plot(list(task_sizes.keys()), probs, marker='o', label=f'τ={temp}')
            ax.legend()
        else:
            loader = MultiTaskDataLoader(
                {name: list(range(size)) for name, size in task_sizes.items()},
                mixing_strategy=strategy
            )
            probs = [loader.sampling_probs[t] * 100 for t in task_sizes.keys()]
            ax.bar(list(task_sizes.keys()), probs)
        
        ax.set_xlabel('Task')
        ax.set_ylabel('Sampling Probability (%)')
        ax.set_title(f'{strategy.capitalize()} Mixing')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('multitask_mixing_strategies.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print actual dataset sizes
    print("\nDataset Sizes:")
    for task, size in task_sizes.items():
        print(f"  {task}: {size:,} examples")

visualize_mixing_strategies()

## 5. Task Interference Detection

In [ ]:
class TaskInterferenceAnalyzer:
    """Detect and measure task interference in multi-task learning."""
    
    def __init__(self, task_names: List[str]):
        self.task_names = task_names
        self.num_tasks = len(task_names)
        
        # Track single-task baselines
        self.single_task_performance = {}
        
        # Track multi-task performance
        self.multi_task_performance = {}
        
        # Gradient similarity matrix
        self.gradient_similarity = np.zeros((self.num_tasks, self.num_tasks))
    
    def compute_gradient_similarity(
        self,
        gradients: Dict[str, torch.Tensor]
    ) -> np.ndarray:
        """
        Compute cosine similarity between task gradients.
        
        Positive similarity: tasks help each other
        Negative similarity: task interference
        """
        grad_vectors = [gradients[task].flatten() for task in self.task_names]
        
        for i, grad_i in enumerate(grad_vectors):
            for j, grad_j in enumerate(grad_vectors):
                similarity = F.cosine_similarity(
                    grad_i.unsqueeze(0),
                    grad_j.unsqueeze(0),
                    dim=1
                ).item()
                self.gradient_similarity[i, j] = similarity
        
        return self.gradient_similarity
    
    def compute_transfer_scores(self) -> Dict[str, float]:
        """
        Compute positive/negative transfer for each task.
        
        Transfer = (Multi-task Performance - Single-task Performance) / Single-task
        """
        transfer_scores = {}
        
        for task in self.task_names:
            if task in self.single_task_performance and task in self.multi_task_performance:
                single = self.single_task_performance[task]
                multi = self.multi_task_performance[task]
                transfer = (multi - single) / single * 100
                transfer_scores[task] = transfer
        
        return transfer_scores
    
    def visualize_interference(self):
        """Visualize task interference patterns."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 1. Gradient similarity heatmap
        ax1 = axes[0]
        im = ax1.imshow(self.gradient_similarity, cmap='RdBu_r', vmin=-1, vmax=1)
        ax1.set_xticks(range(self.num_tasks))
        ax1.set_yticks(range(self.num_tasks))
        ax1.set_xticklabels(self.task_names, rotation=45, ha='right')
        ax1.set_yticklabels(self.task_names)
        ax1.set_title('Gradient Similarity (Interference Detection)')
        
        # Add values to heatmap
        for i in range(self.num_tasks):
            for j in range(self.num_tasks):
                text = ax1.text(j, i, f'{self.gradient_similarity[i, j]:.2f}',
                               ha="center", va="center", color="black", fontsize=9)
        
        plt.colorbar(im, ax=ax1, label='Cosine Similarity')
        
        # 2. Transfer scores
        ax2 = axes[1]
        transfer_scores = self.compute_transfer_scores()
        
        if transfer_scores:
            tasks = list(transfer_scores.keys())
            scores = list(transfer_scores.values())
            colors = ['green' if s > 0 else 'red' for s in scores]
            
            ax2.barh(tasks, scores, color=colors, alpha=0.6)
            ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
            ax2.set_xlabel('Transfer Score (%)')
            ax2.set_title('Positive/Negative Transfer')
            ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('task_interference_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()

# Example: Simulate task interference
def simulate_task_interference():
    """Simulate gradient patterns and interference."""
    task_names = ['Classification', 'NER', 'QA', 'Summarization']
    analyzer = TaskInterferenceAnalyzer(task_names)
    
    # Simulate gradients (normally these would come from actual model)
    np.random.seed(42)
    gradients = {}
    
    # Create correlated gradients for related tasks
    base_grad = torch.randn(1000)
    
    gradients['Classification'] = base_grad + torch.randn(1000) * 0.3
    gradients['NER'] = base_grad + torch.randn(1000) * 0.4  # Similar to classification
    gradients['QA'] = torch.randn(1000)  # Different task
    gradients['Summarization'] = -base_grad + torch.randn(1000) * 0.5  # Negative correlation
    
    analyzer.compute_gradient_similarity(gradients)
    
    # Simulate performance metrics
    analyzer.single_task_performance = {
        'Classification': 0.85,
        'NER': 0.78,
        'QA': 0.72,
        'Summarization': 0.68
    }
    
    analyzer.multi_task_performance = {
        'Classification': 0.87,  # Positive transfer
        'NER': 0.81,  # Positive transfer
        'QA': 0.71,  # Slight negative transfer
        'Summarization': 0.65  # Negative transfer
    }
    
    analyzer.visualize_interference()
    
    # Print analysis
    print("\nTask Interference Analysis:")
    print("\nGradient Similarities:")
    for i, task_i in enumerate(task_names):
        for j, task_j in enumerate(task_names):
            if i < j:
                sim = analyzer.gradient_similarity[i, j]
                print(f"  {task_i} <-> {task_j}: {sim:.3f}")
    
    print("\nTransfer Scores:")
    transfer = analyzer.compute_transfer_scores()
    for task, score in transfer.items():
        status = "Positive" if score > 0 else "Negative"
        print(f"  {task}: {score:+.2f}% ({status} Transfer)")

simulate_task_interference()

## 6. Complete Multi-Task Training System

In [ ]:
class MultiTaskTrainer:
    """Complete multi-task training system with various balancing strategies."""
    
    def __init__(
        self,
        model: nn.Module,
        task_configs: List[TaskConfig],
        weighting_method: str = 'uniform',  # 'uniform', 'uncertainty', 'gradnorm'
        mixing_strategy: str = 'proportional',
        temperature: float = 1.0
    ):
        self.model = model
        self.task_configs = {tc.name: tc for tc in task_configs}
        self.weighting_method = weighting_method
        
        # Initialize weighting mechanism
        if weighting_method == 'uncertainty':
            self.uncertainty_weighting = UncertaintyWeighting(len(task_configs))
        elif weighting_method == 'gradnorm':
            self.gradnorm = GradNormWeighting(len(task_configs))
        
        # Training history
        self.history = defaultdict(list)
        self.task_weights_history = defaultdict(list)
    
    def compute_loss(
        self,
        predictions: Dict[str, torch.Tensor],
        targets: Dict[str, torch.Tensor]
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute weighted multi-task loss.
        
        Returns:
            total_loss: Weighted combination of task losses
            task_losses: Individual task losses (for logging)
        """
        task_losses = {}
        loss_tensor = []
        
        for task_name in predictions.keys():
            config = self.task_configs[task_name]
            pred = predictions[task_name]
            target = targets[task_name]
            
            # Compute task-specific loss
            if config.loss_fn == 'cross_entropy':
                loss = F.cross_entropy(pred, target)
            elif config.loss_fn == 'mse':
                loss = F.mse_loss(pred, target)
            elif config.loss_fn == 'bce':
                loss = F.binary_cross_entropy_with_logits(pred, target)
            else:
                raise ValueError(f"Unknown loss function: {config.loss_fn}")
            
            task_losses[task_name] = loss.item()
            loss_tensor.append(loss)
        
        loss_tensor = torch.stack(loss_tensor)
        
        # Apply weighting strategy
        if self.weighting_method == 'uniform':
            weights = torch.ones(len(loss_tensor)) / len(loss_tensor)
            total_loss = (loss_tensor * weights).sum()
        
        elif self.weighting_method == 'uncertainty':
            total_loss = self.uncertainty_weighting(loss_tensor)
            weights = self.uncertainty_weighting.get_weights()
        
        elif self.weighting_method == 'static':
            weights = torch.tensor([self.task_configs[name].weight 
                                   for name in predictions.keys()])
            total_loss = (loss_tensor * weights).sum()
        
        else:
            weights = torch.ones(len(loss_tensor))
            total_loss = loss_tensor.mean()
        
        # Log weights
        for i, task_name in enumerate(predictions.keys()):
            self.task_weights_history[task_name].append(weights[i].item())
        
        return total_loss, task_losses
    
    def train_step(
        self,
        batch_data: Dict[str, torch.Tensor],
        optimizer: torch.optim.Optimizer
    ) -> Dict[str, float]:
        """Single training step."""
        self.model.train()
        optimizer.zero_grad()
        
        # Forward pass
        predictions = {}
        for task_name, (inputs, targets) in batch_data.items():
            outputs = self.model(inputs, task_name=task_name)
            predictions[task_name] = outputs
        
        # Compute loss
        targets_dict = {task: data[1] for task, data in batch_data.items()}
        total_loss, task_losses = self.compute_loss(predictions, targets_dict)
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        
        # Log losses
        self.history['total_loss'].append(total_loss.item())
        for task_name, loss in task_losses.items():
            self.history[f'loss_{task_name}'].append(loss)
        
        return task_losses
    
    def visualize_training(self):
        """Visualize multi-task training dynamics."""
        fig, axes = plt.subplots(2, 1, figsize=(12, 8))
        
        # 1. Task losses over time
        ax1 = axes[0]
        for task_name in self.task_configs.keys():
            losses = self.history[f'loss_{task_name}']
            if losses:
                # Smooth with moving average
                window = 50
                if len(losses) >= window:
                    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
                    ax1.plot(smoothed, label=task_name, linewidth=2)
        
        ax1.set_xlabel('Training Step')
        ax1.set_ylabel('Loss')
        ax1.set_title('Multi-Task Training: Individual Task Losses')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Task weights over time (for uncertainty/gradnorm)
        ax2 = axes[1]
        for task_name in self.task_configs.keys():
            weights = self.task_weights_history[task_name]
            if weights:
                ax2.plot(weights, label=task_name, linewidth=2)
        
        ax2.set_xlabel('Training Step')
        ax2.set_ylabel('Task Weight')
        ax2.set_title('Dynamic Task Weighting')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('multitask_training_dynamics.png', dpi=300, bbox_inches='tight')
        plt.show()

print("Multi-task training system implemented")

## 7. Practical Multi-Task Examples

In [ ]:
# Example 1: NLP Multi-Task Setup
def create_nlp_multitask_config():
    """Common NLP multi-task configuration."""
    configs = [
        TaskConfig(
            name='sentiment_classification',
            loss_fn='cross_entropy',
            weight=1.0,
            dataset_size=50000,
            num_classes=3
        ),
        TaskConfig(
            name='named_entity_recognition',
            loss_fn='cross_entropy',
            weight=1.0,
            dataset_size=25000,
            num_classes=9  # BIO tags for 4 entity types
        ),
        TaskConfig(
            name='question_answering',
            loss_fn='cross_entropy',
            weight=1.5,  # Higher weight for harder task
            dataset_size=10000,
            num_classes=512  # Span start/end
        ),
        TaskConfig(
            name='paraphrase_detection',
            loss_fn='bce',
            weight=1.0,
            dataset_size=30000,
            num_classes=1
        )
    ]
    
    return configs

# Visualize task relationships
def visualize_task_relationships():
    """Visualize expected task synergies."""
    tasks = ['Sentiment', 'NER', 'QA', 'Paraphrase']
    
    # Hypothetical synergy matrix (based on shared skills)
    synergy = np.array([
        [1.0, 0.6, 0.4, 0.7],  # Sentiment
        [0.6, 1.0, 0.8, 0.5],  # NER
        [0.4, 0.8, 1.0, 0.6],  # QA
        [0.7, 0.5, 0.6, 1.0]   # Paraphrase
    ])
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(synergy, cmap='YlOrRd', vmin=0, vmax=1)
    
    ax.set_xticks(range(len(tasks)))
    ax.set_yticks(range(len(tasks)))
    ax.set_xticklabels(tasks, rotation=45, ha='right')
    ax.set_yticklabels(tasks)
    ax.set_title('Expected Task Synergy Matrix')
    
    # Add values
    for i in range(len(tasks)):
        for j in range(len(tasks)):
            text = ax.text(j, i, f'{synergy[i, j]:.1f}',
                          ha="center", va="center", color="black")
    
    plt.colorbar(im, ax=ax, label='Synergy Score')
    plt.tight_layout()
    plt.savefig('task_synergy_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

configs = create_nlp_multitask_config()
print("\nNLP Multi-Task Configuration:")
for config in configs:
    print(f"\n{config.name}:")
    print(f"  Loss: {config.loss_fn}")
    print(f"  Weight: {config.weight}")
    print(f"  Dataset size: {config.dataset_size:,}")
    print(f"  Classes: {config.num_classes}")

visualize_task_relationships()

## 8. Multi-Task Evaluation

In [ ]:
class MultiTaskEvaluator:
    """Comprehensive multi-task evaluation."""
    
    def __init__(self, task_names: List[str]):
        self.task_names = task_names
        self.results = defaultdict(dict)
    
    def evaluate(
        self,
        model: nn.Module,
        test_loaders: Dict[str, any],
        metrics: Dict[str, callable]
    ) -> Dict[str, Dict[str, float]]:
        """
        Evaluate model on all tasks.
        
        Args:
            model: Multi-task model
            test_loaders: Dict mapping task names to data loaders
            metrics: Dict mapping task names to metric functions
        """
        model.eval()
        results = {}
        
        with torch.no_grad():
            for task_name in self.task_names:
                if task_name not in test_loaders:
                    continue
                
                predictions = []
                targets = []
                
                for batch in test_loaders[task_name]:
                    inputs, labels = batch
                    outputs = model(inputs, task_name=task_name)
                    predictions.append(outputs)
                    targets.append(labels)
                
                predictions = torch.cat(predictions)
                targets = torch.cat(targets)
                
                # Compute task-specific metrics
                metric_fn = metrics.get(task_name, self._default_metric)
                score = metric_fn(predictions, targets)
                results[task_name] = score
        
        return results
    
    def _default_metric(self, predictions: torch.Tensor, targets: torch.Tensor) -> float:
        """Default accuracy metric."""
        pred_labels = predictions.argmax(dim=-1)
        accuracy = (pred_labels == targets).float().mean().item()
        return accuracy
    
    def compare_single_vs_multi(
        self,
        single_task_results: Dict[str, float],
        multi_task_results: Dict[str, float]
    ):
        """Compare single-task vs multi-task performance."""
        fig, ax = plt.subplots(figsize=(10, 6))
        
        tasks = list(single_task_results.keys())
        x = np.arange(len(tasks))
        width = 0.35
        
        single_scores = [single_task_results[t] for t in tasks]
        multi_scores = [multi_task_results[t] for t in tasks]
        
        ax.bar(x - width/2, single_scores, width, label='Single-Task', alpha=0.8)
        ax.bar(x + width/2, multi_scores, width, label='Multi-Task', alpha=0.8)
        
        # Add improvement percentages
        for i, task in enumerate(tasks):
            improvement = (multi_scores[i] - single_scores[i]) / single_scores[i] * 100
            y_pos = max(single_scores[i], multi_scores[i]) + 0.02
            color = 'green' if improvement > 0 else 'red'
            ax.text(i, y_pos, f'{improvement:+.1f}%', ha='center', 
                   color=color, fontweight='bold')
        
        ax.set_xlabel('Task')
        ax.set_ylabel('Performance Score')
        ax.set_title('Single-Task vs Multi-Task Performance')
        ax.set_xticks(x)
        ax.set_xticklabels(tasks, rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig('single_vs_multi_task.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print summary
        print("\nPerformance Comparison:")
        print(f"{'Task':<20} {'Single':<10} {'Multi':<10} {'Change'}")
        print("-" * 50)
        
        avg_single = 0
        avg_multi = 0
        
        for task in tasks:
            single = single_task_results[task]
            multi = multi_task_results[task]
            change = (multi - single) / single * 100
            print(f"{task:<20} {single:.4f}    {multi:.4f}    {change:+.2f}%")
            avg_single += single
            avg_multi += multi
        
        avg_single /= len(tasks)
        avg_multi /= len(tasks)
        avg_change = (avg_multi - avg_single) / avg_single * 100
        
        print("-" * 50)
        print(f"{'Average':<20} {avg_single:.4f}    {avg_multi:.4f}    {avg_change:+.2f}%")

# Example comparison
evaluator = MultiTaskEvaluator(['Sentiment', 'NER', 'QA', 'Paraphrase'])

# Simulated results
single_task = {
    'Sentiment': 0.872,
    'NER': 0.845,
    'QA': 0.723,
    'Paraphrase': 0.791
}

multi_task = {
    'Sentiment': 0.885,  # +1.5%
    'NER': 0.862,  # +2.0%
    'QA': 0.718,  # -0.7%
    'Paraphrase': 0.804  # +1.6%
}

evaluator.compare_single_vs_multi(single_task, multi_task)

## 9. Best Practices & Recommendations

### Task Selection

**Good Task Combinations:**
- Related tasks with shared linguistic features (NER + POS tagging)
- Different granularities of same problem (sentence + document classification)
- Complementary skills (classification + generation)

**Avoid:**
- Completely unrelated tasks (may cause negative transfer)
- Tasks with conflicting objectives
- Extreme imbalance in task difficulty

### Dataset Mixing Guidelines

1. **For balanced tasks:** Use uniform or proportional sampling
2. **For imbalanced tasks:** Use temperature-scaled sampling ($\tau = 0.5$)
3. **For rare tasks:** Set artificial maximums on large datasets
4. **For high-value tasks:** Manually increase task weight

### Loss Weighting Strategy Selection

| Method | Use When | Pros | Cons |
|--------|----------|------|------|
| Uniform | Tasks well-balanced | Simple, stable | May underperform |
| Uncertainty | Heterogeneous tasks | Automatic, theoretically grounded | Requires tuning |
| GradNorm | Dynamic task difficulty | Balances learning rates | Complex, computationally expensive |
| Manual | Domain knowledge available | Full control | Requires expertise |

### Training Tips

1. **Start with single-task baselines** to establish performance targets
2. **Monitor task interference** using gradient similarity
3. **Use gradual task introduction** for many tasks (curriculum learning)
4. **Implement task-specific learning rates** if needed
5. **Regular evaluation** on all tasks (not just total loss)
6. **Task tokens/prefixes** help model distinguish tasks

### Architecture Considerations

**Hard Parameter Sharing:**
- Shared encoder, task-specific heads
- Best for: related tasks, limited compute
- Risk: negative transfer

**Soft Parameter Sharing:**
- Separate networks with regularization
- Best for: loosely related tasks
- Risk: more parameters to train

**Task-Specific Adapters:**
- Shared backbone + task adapters
- Best for: many tasks, parameter efficiency
- Combines benefits of both approaches

### Common Pitfalls

1. **Ignoring task interference:** Monitor and address negative transfer
2. **Poor dataset mixing:** Rare tasks get insufficient training
3. **Evaluation on total loss only:** Individual task performance may degrade
4. **Forgetting single-task baselines:** Can't measure actual benefit
5. **Too many disparate tasks:** Consider task clustering or routing

## 10. Benchmarks & Real-World Results

### Academic Benchmarks

**GLUE Benchmark (Multi-Task):**
- MT-DNN: 82.3 → 84.7 with multi-task pre-training
- Tasks: CoLA, SST-2, MRPC, QQP, MNLI, QNLI, RTE, STS-B

**SuperGLUE:**
- T5: Unified text-to-text framework
- Single model for all tasks: 89.3 average

**decaNLP (10 NLP Tasks):**
- MQAN: Question answering framework for all tasks
- Demonstrates multi-task as unified QA

### Industry Applications

**Google T5:**
- Pre-trained on mixture of unsupervised and supervised tasks
- C4 dataset + supervised mixture
- Powers many Google products

**Meta's BART:**
- Multi-task denoising
- Strong performance on generation tasks

**OpenAI InstructGPT:**
- Instruction tuning as multi-task learning
- Thousands of instruction-following tasks

### Performance Patterns

**Typical Improvements:**
- Low-resource tasks: +5-15% from multi-task learning
- High-resource tasks: +0-5% (sometimes slight degradation)
- Related task groups: Better transfer than diverse tasks

**Resource Requirements:**
- Training time: 1.2-1.8x single-task (depends on mixing)
- Memory: Constant (shared encoder) to 2x (task-specific layers)
- Inference: Minimal overhead with proper task routing

## Summary

Multi-task fine-tuning enables:
1. **Knowledge transfer** between related tasks
2. **Improved generalization** through shared representations
3. **Parameter efficiency** with single shared model
4. **Better low-resource performance** via task synergy

**Key Takeaways:**
- Task selection and balancing are critical
- Monitor for task interference
- Use appropriate mixing strategies for dataset imbalance
- Consider uncertainty or gradient-based weighting
- Always compare against single-task baselines

**Next Steps:**
- Experiment with different weighting strategies
- Implement task-specific adapters
- Try mixture-of-experts for task routing (Notebook 49)
- Address catastrophic forgetting in continual learning (Notebook 46)